# MAP5935 - Statistical Learning (Chapter 12 - Unsupervised Learning)

**Prof. Christian Jäkel**

https://www.statlearning.com/

In [4]:
import numpy as np
import random
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.pyplot import subplots

from statsmodels.datasets import get_rdataset
from sklearn.metrics import pairwise_distances

from ISLP import load_data
from ISLP.models import ModelSpec as MS

## *Conceptual Exercises*

### (4) Suppose that for a particular data set, we perform hierarchical clustering using single linkage and using complete linkage. We obtain two dendrograms.

### (a) At a certain point on the single linkage dendrogram, the clusters \\(\{1, 2, 3\}\\) and \\(\{4, 5\}\\) fuse. On the complete linkage dendrogram, the clusters \\(\{1, 2, 3\}\\) and \\(\{4, 5\}\\) also fuse at a certain point. Which fusion will occur higher on the tree, or will they fuse at the same height, or is there not enough information to tell?

In page 529, we have that *The concept of dissimilarity between a pair of observations needs to be extended to a pair of groups of observations. This extension is achieved by developing the notion of linkage, which defines the dissimilarity between two groups of observations. The four most common types of linkage—**complete**, **average**, **single**, linkage and centroid are briefly described in Table 12.3. Average, complete, and single linkage are most popular among statisticians.*

From the table we have that:

- **Single** - Minimal intercluster dissimilarity. Compute all pairwise dissimilarities between the observations in cluster A and the observations in cluster B, and record the smallest of these dissimilarities. Single linkage can result in extended, trailing clusters in which single observations are fused one-at-a-time.
- **Complete** - Maximal intercluster dissimilarity. Compute all pairwise dissimilarities between the observations in cluster A and the observations in cluster B, and record the largest of these dissimilarities.

In other words we can write:

- **Single linkage:** distance between clusters \\(A\\) and \\(B\\) is  
  \\(\displaystyle D_{\text{single}}(A,B) = \min_{i \in A,\; j \in B} d(i,j)\\).
- **Complete linkage:** distance between clusters \\(A\\) and \\(B\\) is  
  \\(\displaystyle D_{\text{complete}}(A,B) = \max_{i \in A,\; j \in B} d(i,j)\\).

Let \\(d(i,j)\\) denote the dissimilarity between observations \\(i\\) and \\(j\\).

Since \\(\max \ge \min\\) for any set of numbers, we always have  
\\(D_{\text{complete}}(A,B) \ge D_{\text{single}}(A,B)\\).

For the fusion of clusters \\(\{1,2,3\}\\) and \\(\{4,5\}\\):

- Under **single linkage**, the height is  
  \\(h_{\text{single}} = \min_{i \in \{1,2,3\},\, j \in \{4,5\}} d(i,j)\\).
  
- Under **complete linkage**, the height is  
  \\(h_{\text{complete}} = \max_{i \in \{1,2,3\},\, j \in \{4,5\}} d(i,j)\\).

Hence \\(h_{\text{complete}} \ge h_{\text{single}}\\).

So, in our case, the fusion of \\(\{1,2,3\}\\) and \\(\{4,5\}\\) will occur **higher on the complete linkage dendrogram** (it could coincide only in the special case where all six intercluster distances are equal, but it can never be lower).

### (b) At a certain point on the single linkage dendrogram, the clusters \\(\{5\}\\) and \\(\{6\}\\) fuse. On the complete linkage dendrogram, the clusters \\(\{5\}\\) and \\(\{6\}\\) also fuse at a certain point. Which fusion will occur higher on the tree, or will they fuse at the same height, or is there not enough information to tell?

For the fusion of clusters \\(\{5\}\\) and \\(\{6\}\\):

- There is only one pair of points, so  
  \\(D_{\text{single}}(\{5\},\{6\}) = \min_{i \in \{5\},\, j \in \{6\}} d(5,6) = d(5,6)\\),
- and  
  \\(D_{\text{complete}}(\{5\},\{6\}) = \max_{i \in \{5\},\, j \in \{6\}} d(5,6) = d(5,6)\\).

Thus both methods use the **same** intercluster distance, so the fusion of \\(\{5\}\\) and \\(\{6\}\\) will occur **at the same height on both dendrograms**.

## *Applied Exercises*

### (7) In this chapter, we mentioned the use of correlation-based distance and Euclidean distance as dissimilarity measures for hierarchical clustering. It turns out that these two measures are almost equivalent: if each observation has been centered to have mean zero and standard deviation one, and if we let \\(r_{ij}\\) denote the correlation between the \\(i\\)-th and \\(j\\)-th observations, then the quantity \\(1−r_{ij}\\) is proportional to the squared Euclidean distance between the ith and jth observations. On the `USArrests` data, show that this proportionality holds.

### **Hint**: The Euclidean distance can be calculated using the `pairwise_distances()` function from the `sklearn.metrics` module, and `pairwise_` correlations can be calculated using the `np.corrcoef()` function.

In [13]:
USArrests = get_rdataset('USArrests').data
USArrests.info()

<class 'pandas.core.frame.DataFrame'>
Index: 50 entries, Alabama to Wyoming
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Murder    50 non-null     float64
 1   Assault   50 non-null     int64  
 2   UrbanPop  50 non-null     int64  
 3   Rape      50 non-null     float64
dtypes: float64(2), int64(2)
memory usage: 2.0+ KB


### Relationship between correlation-based distance and squared Euclidean distance

We want to show that, after standardizing each **observation** (row) to have
mean 0 and standard deviation 1, the correlation-based dissimilarity  
\\(1 - r_{ij}\\) between observations \\(i\\) and \\(j\\) is **proportional** to the
**squared Euclidean distance** between them, \\(D_{ij}^2\\).

---

#### 1. Row standardization

Let \\(X\\) be the \\(n \times p\\) data matrix (here \\(n=50\\) states,
\\(p=4\\) variables: Murder, Assault, UrbanPop, Rape).  
Row \\(i\\) is the vector

\\[
\mathbf{x}_i = (x_{i1}, x_{i2}, \ldots, x_{ip})^\top .
\\]

We standardize **each row**:

\\[
\tilde{\mathbf{x}}_i
= \frac{\mathbf{x}_i - \bar{x}_i \mathbf{1}}{s_i},
\\]

where

- \\(\bar{x}_i = \dfrac{1}{p} \sum_{k=1}^p x_{ik}\\) is the mean of row \\(i\\),
- \\(s_i\\) is the sample standard deviation of row \\(i\\),

\\[
s_i^2 = \frac{1}{p-1} \sum_{k=1}^p (x_{ik} - \bar{x}_i)^2 .
\\]

After this transformation, **each row has mean 0 and variance 1**:

\\[
\frac{1}{p} \sum_{k=1}^p \tilde{x}_{ik} = 0,
\qquad
\frac{1}{p-1} \sum_{k=1}^p \tilde{x}_{ik}^2 = 1.
\\]

From the variance condition we obtain

\\[
\sum_{k=1}^p \tilde{x}_{ik}^2 = p-1
\quad \text{for every } i.
\\]

(Here the tilde denotes the standardized rows.)

---

#### 2. Squared Euclidean distance between observations

For two standardized observations \\(\tilde{\mathbf{x}}_i\\) and
\\(\tilde{\mathbf{x}}_j\\), the **squared Euclidean distance** is

\\[
D_{ij}^2
= \sum_{k=1}^p (\tilde{x}_{ik} - \tilde{x}_{jk})^2.
\\]

Using the identity \\((a-b)^2 = a^2 + b^2 - 2ab\\), we expand:

\\[
\begin{aligned}
D_{ij}^2
&= \sum_{k=1}^p \left( \tilde{x}_{ik}^2 + \tilde{x}_{jk}^2 - 2 \tilde{x}_{ik}\tilde{x}_{jk} \right) \\
&= \underbrace{\sum_{k=1}^p \tilde{x}_{ik}^2}_{=\,p-1}
   + \underbrace{\sum_{k=1}^p \tilde{x}_{jk}^2}_{=\,p-1}
   - 2 \sum_{k=1}^p \tilde{x}_{ik}\tilde{x}_{jk}.
\end{aligned}
\\]

Therefore

\\[
D_{ij}^2
= 2(p-1) - 2\sum_{k=1}^p \tilde{x}_{ik}\tilde{x}_{jk}.
\\]

---

#### 3. Correlation between observations (states)

Since each row has mean 0 and variance 1, the **sample correlation** between
rows \\(i\\) and \\(j\\) is

\\[
r_{ij}
= \frac{\mathrm{Cov}(\tilde{\mathbf{x}}_i,\tilde{\mathbf{x}}_j)}{\sqrt{\mathrm{Var}(\tilde{\mathbf{x}}_i)\,\mathrm{Var}(\tilde{\mathbf{x}}_j)}}
= \mathrm{Cov}(\tilde{\mathbf{x}}_i,\tilde{\mathbf{x}}_j),
\\]

because both variances are 1.

The sample covariance is

\\[
\mathrm{Cov}(\tilde{\mathbf{x}}_i,\tilde{\mathbf{x}}_j)
= \frac{1}{p-1} \sum_{k=1}^p \tilde{x}_{ik}\tilde{x}_{jk}.
\\]

Hence

\\[
\sum_{k=1}^p \tilde{x}_{ik}\tilde{x}_{jk} = (p-1)\,r_{ij}.
\\]

Plugging this into the expression for \\(D_{ij}^2\\):

\\[
\begin{aligned}
D_{ij}^2
&= 2(p-1) - 2 (p-1) r_{ij} \\
&= 2(p-1)\bigl(1 - r_{ij}\bigr).
\end{aligned}
\\]

We obtain the **exact theoretical relationship**

\\[
\boxed{D_{ij}^2 = 2(p-1)\bigl(1 - r_{ij}\bigr)},
\\]

so squared Euclidean distance and correlation-based dissimilarity
\\(1 - r_{ij}\\) are proportional, with proportionality constant \\(k = 2(p-1)\\).

For the `USArrests` data set we have \\(p = 4\\), so

\\[
k = 2(p-1) = 2 \times 3 = 6,
\\]

and therefore

\\[
D_{ij}^2 = 6 \, (1 - r_{ij}).
\\]

In [5]:
# ---------------------------------------------------------------------
# 1. Load the USArrests data
#    (adjust the path/file name below to where your csv is located)
# ---------------------------------------------------------------------
df = USArrests

X = df.values.astype(float)          # shape: (50 states, 4 variables)
n, p = X.shape

# ---------------------------------------------------------------------
# 2. Standardize *each observation* (row) to mean 0 and sd 1
# ---------------------------------------------------------------------
row_means = X.mean(axis=1, keepdims=True)
row_stds  = X.std(axis=1, ddof=1, keepdims=True)
X_std = (X - row_means) / row_stds   # each row: mean 0, sd 1

# ---------------------------------------------------------------------
# 3. Squared Euclidean distances between observations
# ---------------------------------------------------------------------
D = pairwise_distances(X_std, metric="euclidean")
D2 = D ** 2                          # squared Euclidean distances

# ---------------------------------------------------------------------
# 4. Correlations between observations (states)
#    np.corrcoef with default rowvar=True treats each row as a variable,
#    so here we get a 50x50 matrix of correlations between states.
# ---------------------------------------------------------------------
R = np.corrcoef(X_std)               # shape: (50, 50)

one_minus_R = 1 - R

# ---------------------------------------------------------------------
# 5. Show proportionality: D2_ij ≈ k * (1 - r_ij)
#    We estimate k using least squares on off-diagonal elements.
# ---------------------------------------------------------------------
mask = ~np.eye(n, dtype=bool)        # mask to drop diagonal entries

D2_vec = D2[mask]
one_minus_R_vec = one_minus_R[mask]

k_hat = np.sum(D2_vec * one_minus_R_vec) / np.sum(one_minus_R_vec ** 2)
print(f"Estimated proportionality constant k ≈ {k_hat:.4f}")

# Theoretical constant when each row is standardized:
#   D^2_ij = 2 (p - 1) (1 - r_ij)
print("Theoretical 2*(p-1) =", 2 * (p - 1))

# Check how close the relationship is numerically
max_diff = np.max(np.abs(D2_vec - k_hat * one_minus_R_vec))
print(f"Max |D^2_ij - k*(1 - r_ij)| over i≠j: {max_diff:.6e}")


Estimated proportionality constant k ≈ 6.0000
Theoretical 2*(p-1) = 6
Max |D^2_ij - k*(1 - r_ij)| over i≠j: 3.105155e-15


#### 4. Conclusion

When we verify numerically that \\(D_{ij}^2 \approx 2(p-1)\bigl(1 - r_{ij}\bigr)\\) for all pairs \\(i \neq j\\), i.e., the discrepancies are only due to floating-point rounding error, we confirm on the `USArrests` data that the correlation-based dissimilarity \\(1 - r_{ij}\\) is indeed proportional to the squared Euclidean distance between observations, once each observation has been centered to mean 0 and scaled to standard deviation 1.
